## ****Default Lakehouse:****`lh_Gold_StratusCore`
**Read from:** `lh_Silver_StratusCore` (via abfss:// path)

****Kimball Methodology Applied:****
> - Conformed dimensions shared across both subject areas (Availability + Outage)
> - Surrogate keys (integer) on all dimension tables
> - Natural/business keys preserved in dimensions for traceability
> - Fact tables contain only foreign keys (surrogate) and additive measures
> - Aggregate tables pre-computed for Power BI performance


### Cell 1 — Imports and Config


In [6]:
from pyspark.sql import SparkSession
from datetime import datetime

spark = SparkSession.builder.getOrCreate()

SILVER_BASE ="abfss://Swift@onelake.dfs.fabric.microsoft.com/lh_Silver_StratusCoreTelecoms.Lakehouse/Tables"

# Read from Silver
sdf_avail = spark.table(
    "lh_Silver_StratusCoreTelecoms.dbo.silver_stratus_availability"
)



# Add reporting columns
sdf_gold_avail = sdf_avail.withColumn(
    "is_sla_breach",
    when(col("Availability") < 99.5, True).otherwise(False)
).withColumn(
    "month_label",
    date_format(col("date"), "MMM yyyy")
    )

# Write to Gold
sdf_gold_avail.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("gold_stratus_availability")



StatementMeta(, 262d1cd4-8d69-4c0e-960c-bfc6fa1f708c, 8, Finished, Available, Finished, False)

In [10]:
# Read from Silver
sdf_outage = spark.table(
    "lh_Silver_StratusCoreTelecoms.dbo.silver_stratus_outage"
)

# Transform
sdf_gold_outage = sdf_outage.withColumn(
    "outage_year", year(col("Date"))
).withColumn(
    "outage_month", month(col("Date"))
).withColumn(
    "duration_hours", fround(col("Outage_Duration_Mins") / 60, 2)
)

# Write to Gold
sdf_gold_outage.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("gold_stratus_outage")

StatementMeta(, 262d1cd4-8d69-4c0e-960c-bfc6fa1f708c, 12, Finished, Available, Finished, False)